# Description

## Description du data set
- Les données sont décomposées en 3 datasets (Train, Validation, Test)
- Chaque ligne représente une question, et 5 réponses au choix. Une seule réponse correct par question
- La colonne "correct_answers" est la validation de la bonne réponse (allant de 0 à 4)

In [2]:
from google.cloud import storage
from datasets import load_from_disk
import gcsfs
import pandas as pd

# # Authentification (si pas déjà fait)
# from google.colab import auth
# auth.authenticate_user()

c:\Users\Fabien\Desktop\OC\P14\FINE-TUNING_MEDICAL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Chargement direct depuis le bucket, sans copie locale
dataset = load_from_disk("gs://p14-medical-data/raw_data/frenchmedmcqa_dataset/")
# Vérification rapide
print(dataset)
print(dataset["train"][0])  # Premier exemple

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'answer_a', 'answer_b', 'answer_c', 'answer_d', 'answer_e', 'correct_answers', 'number_correct_answers'],
        num_rows: 595
    })
    validation: Dataset({
        features: ['id', 'question', 'answer_a', 'answer_b', 'answer_c', 'answer_d', 'answer_e', 'correct_answers', 'number_correct_answers'],
        num_rows: 164
    })
    test: Dataset({
        features: ['id', 'question', 'answer_a', 'answer_b', 'answer_c', 'answer_d', 'answer_e', 'correct_answers', 'number_correct_answers'],
        num_rows: 321
    })
})
{'id': '230bac49b0fe863b772410bc8d01a025f63c3c999065480131d6334abd2efeff', 'question': 'Parmi les affirmations suivantes, une seule est fausse, indiquer laquelle: les particules alpha', 'answer_a': "Sont formées de noyaux d'hélium", 'answer_b': 'Sont peu pénétrantes', 'answer_c': "Toute l'énergie qu'elles transportent est cédée au long d'un parcours de quelques centimètres dans l'air", 'answer_d':

# Analysis dataset

## Exploration dataset train

In [4]:
df = pd.DataFrame(dataset["train"])
df.head(5)

,id,question,answer_a,answer_b,answer_c,answer_d,answer_e,correct_answers,number_correct_answers
0,230bac49b0fe863b772410bc8d01a025f63c3c99906548...,"Parmi les affirmations suivantes, une seule es...",Sont formées de noyaux d'hélium,Sont peu pénétrantes,Toute l'énergie qu'elles transportent est cédé...,Sont arrêtées par une feuille de papier,Sont peu ionisantes,4,0
1,0ca718ff033bb68503e41c4d30b4f7580cb3f220e000a2...,"Parmi les bactéries suivantes, une seule ne pe...",Haemophilus influenzae,Streptococcus pneumoniae,Neisseria gonorrhoeae,Neisseria meningitidis,Mycobacterium tuberculosis,2,0
2,3e292fc26ab1844f7d3805a5cceba0534261fcd65446dc...,"Parmi les propositions suivantes, indiquer cel...",D'héroine,De cannabis,De cocaïne,D'amphétamines,De LSD (Amide de l'acide lysergique),2,0
3,6c35039202449b2b89e7b8a79dd0d7ae1b73ad65098844...,"Parmi les propositions suivantes, une seule es...",5-hydroxy tryptophane,5-hydroxy tryptamine,Dihydroxy phénylalanine,Dihydroxy tryptophane,"5,6-dihydroxytryptamine",1,0
4,dd39598f83a51258dc16cd37c21496ed298cf66b7245f8...,"Parmi les techniques voltampérométriques, on t...",La potentiométrie,La polarographie,La conductimétrie,l'ampérométrie,La coulométrie,1,0


In [5]:
df.describe(include='all')

,id,question,answer_a,answer_b,answer_c,answer_d,answer_e,correct_answers,number_correct_answers
count,595,595,595,595,595,595,595,595.000000,595.0
unique,594,592,574,576,572,578,541,NaN,NaN
top,04c58320a365b34e9e7fc96b9b969731265be6e71453cc...,"Parmi les propositions suivantes, quelle est c...",Il s'agit d'un virus à ARN,Il appartient à la famille des Hepadnaviridae,Il possède une enveloppe lipoprotéique,Erythromycine,Aucune de ces réponses n'est exacte,NaN,NaN
freq,2,2,3,3,3,3,33,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.028571,0.0
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.288720,0.0
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.0
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.0
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.000000,0.0
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.000000,0.0


In [6]:
df["correct_answers"].value_counts()

correct_answers
3    156
2    142
1    119
0     94
4     84
Name: count, dtype: int64

In [7]:
df.isna().sum()

id                        0
question                  0
answer_a                  0
answer_b                  0
answer_c                  0
answer_d                  0
answer_e                  0
correct_answers           0
number_correct_answers    0
dtype: int64

## Colonnes avec doublons

In [8]:
df[df.duplicated(keep=False)]

,id,question,answer_a,answer_b,answer_c,answer_d,answer_e,correct_answers,number_correct_answers
292,04c58320a365b34e9e7fc96b9b969731265be6e71453cc...,"Parmi les caractères suivants, lequel n'est pa...",Présence d'une capsule,Catalase négative,Survie en aérobiose et anaérobiose,Production d'une toxine de Panton-Valentine,Hémolytique sur gélose au sang,3,0
301,04c58320a365b34e9e7fc96b9b969731265be6e71453cc...,"Parmi les caractères suivants, lequel n'est pa...",Présence d'une capsule,Catalase négative,Survie en aérobiose et anaérobiose,Production d'une toxine de Panton-Valentine,Hémolytique sur gélose au sang,3,0


In [9]:
df[df.duplicated(subset=['question'])]

,id,question,answer_a,answer_b,answer_c,answer_d,answer_e,correct_answers,number_correct_answers
168,b8d72196ad0863e712a0a4fd827a0aae5773af49b04cfe...,"Parmi les propositions suivantes, quelle est c...",Tous les établissements de santé peuvent rétro...,Seuls les établissements de santé publics peuv...,Seuls les établissements autorisés peuvent rét...,Tous les médicaments peuvent être rétrocédés,Seuls les médicaments réservés à l'usage hospi...,2,0
293,0835a193cf102df1c723e22f929d29f511c7fc10c253d3...,"Parmi ces propositions, laquelle est fausse ?",Il existe une région homologue dans les gonoso...,Dans les maladies génétiques à transmission do...,Toutes les filles d'un homme atteint de maladi...,La transmission des maladies génétiques monogé...,Les ovocytes sont des cellules haploïdes.,3,0
301,04c58320a365b34e9e7fc96b9b969731265be6e71453cc...,"Parmi les caractères suivants, lequel n'est pa...",Présence d'une capsule,Catalase négative,Survie en aérobiose et anaérobiose,Production d'une toxine de Panton-Valentine,Hémolytique sur gélose au sang,3,0


**ATTENTION A L'INTERPRETATION DES QUESTIONS REPONSES COMME CELLES CI**

## Data Cleaning

In [10]:
df = df.drop(columns=["id", "number_correct_answers" ])

In [11]:
match_dict = {
    0 : "answer_a",
    1 : "answer_b",
    2 : "answer_c",
    3 : "answer_d",
    4 : "answer_e"
}
df['correct_answer_text'] = df['correct_answers'].map(match_dict)

In [12]:
df['answer'] = df.apply(lambda row: row[row['correct_answer_text']],axis=1)

In [13]:
df_cleaned = df.loc[:,['question', 'answer']]

In [14]:
df_cleaned.head()

,question,answer
0,"Parmi les affirmations suivantes, une seule es...",Sont peu ionisantes
1,"Parmi les bactéries suivantes, une seule ne pe...",Neisseria gonorrhoeae
2,"Parmi les propositions suivantes, indiquer cel...",De cocaïne
3,"Parmi les propositions suivantes, une seule es...",5-hydroxy tryptamine
4,"Parmi les techniques voltampérométriques, on t...",La polarographie


In [15]:
df_cleaned[df_cleaned.duplicated(keep=False)]

,question,answer
292,"Parmi les caractères suivants, lequel n'est pa...",Production d'une toxine de Panton-Valentine
301,"Parmi les caractères suivants, lequel n'est pa...",Production d'une toxine de Panton-Valentine


In [17]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
df_cleaned.head(100)

,question,answer
0,"Parmi les affirmations suivantes, une seule est fausse, indiquer laquelle: les particules alpha",Sont peu ionisantes
1,"Parmi les bactéries suivantes, une seule ne peut généralement pas être responsable d'une méningite aiguë, laquelle?",Neisseria gonorrhoeae
2,"Parmi les propositions suivantes, indiquer celle qui est exacte. Le crack est une forme:",De cocaïne
3,"Parmi les propositions suivantes, une seule est exacte. Laquelle? La sérotonine est le (la) :",5-hydroxy tryptamine
4,"Parmi les techniques voltampérométriques, on trouve:",La polarographie
5,"Concernant le misoprostol (CYTOTEC®), donner son mécanisme d'action:",Analogue des prostaglandines
6,Quelle est la proposition qui ne s'applique pas à la norfloxacine ?,Elle est utilisable pour le traitement des broncho-pneumopathies à bactérie sensible
7,"Parmi les propositions suivantes concernant la fluorimétrie, une seule proposition est fausse. Laquelle?",La fluorescence est un phénomène dont la durée est de l'ordre de la seconde
8,La sécrétion de progestérone pendant les 2e et 3e trimestres de la grossesse reflète l'état fonctionnel:,Du placenta
9,"Parmi les propositions suivantes, une seule est exacte; laquelle? L'hormone thyroïdienne la plus active au niveau cellulaire est:",La triiodothyronine (T3)


# cleaned data Visualisation 

In [37]:
pd.set_option('display.max_colwidth', False)

df = pd.read_parquet("gs://p14-medical-data/processed_data/frenchmedmcqa_dataset/frenchmedmcqa_train.parquet")
df.head()

,question,answer
0,"Parmi les affirmations suivantes, une seule est fausse, indiquer laquelle: les particules alpha",Sont peu ionisantes
1,"Parmi les bactéries suivantes, une seule ne peut généralement pas être responsable d'une méningite aiguë, laquelle?",Neisseria gonorrhoeae
2,"Parmi les propositions suivantes, indiquer celle qui est exacte. Le crack est une forme:",De cocaïne
3,"Parmi les propositions suivantes, une seule est exacte. Laquelle? La sérotonine est le (la) :",5-hydroxy tryptamine
4,"Parmi les techniques voltampérométriques, on trouve:",La polarographie
